# Data splits 
The strategy, as suggested by ChatGPT:


1. Split the data into train / validation / test.

2. Put test aside completely.

3. For each model family:

   - Logistic regression
   - Random forest
   - SVM
   - Gradient boosting
   - etc.

   - Use **only the training set** to:
       - tune hyperparameters with K-fold CV;
       - choose class-specific thresholds;
       - estimate fold-to-fold variability.

4. Evaluate each trained model on the **validation** set.

5. **Use validation-set results** to:
       - inspect false positives / false negatives;
       - identify problematic classes;
       - **decide what to try next**.

6. Once the **final** modeling strategy is fixed:
       - optionally combine train + validation;
       - redo threshold/hyperparameter selection using CV on train + validation;
       - train the final model on train + validation.

7. **Evaluate once on the untouched test set**.

In [7]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

---
#### Loading the data

In [8]:
#df = pd.read_parquet('/content/drive/MyDrive/SB/SB_Project/classification_ring/data/processed/data_ml.parquet')
df = pd.read_parquet('./classification_ring/data/processed/data_ml.parquet')

df.head()

,pdb_id,s_ch,s_resi,s_ins,s_resn,s_ss8,s_rsa,s_phi,s_psi,s_a1,...,t_a5,t_3di_state,t_3di_letter,HBOND,IONIC,PICATION,PIHBOND,PIPISTACK,SSBOND,VDW
0,1u9c,A,172,,S,H,0.354,-1.001,-0.749,-0.228,...,-0.912,17.0,R,1,0,0,0,0,0,1
1,1u9c,A,184,,G,-,0.250,-2.102,-3.018,-0.384,...,-1.853,19.0,T,1,0,0,0,0,0,0
2,1u9c,A,10,,T,-,0.000,-1.252,2.751,-0.032,...,-3.242,12.0,M,1,0,0,0,0,0,0
3,1u9c,A,133,,V,T,0.021,-1.076,-0.664,-1.337,...,1.313,15.0,P,0,0,0,0,0,0,1
4,1u9c,A,146,,G,T,0.548,1.315,0.127,-0.384,...,2.064,6.0,G,1,0,0,0,0,0,0


In [9]:
#print(df.columns.tolist())
#print(df.dtypes)
#looking at the column names

pair_cols = [
    "pdb_id",
    "s_ch", "s_resi", "s_ins", "s_resn",
    "t_ch", "t_resi", "t_ins", "t_resn"
]

label_cols = ['HBOND', 'VDW', 'IONIC', 'PIPISTACK', 'PICATION', 'SSBOND', 'PIHBOND']

#Numerical features
num_features = [
    's_rsa', 's_phi', 's_psi', 's_a1', 's_a2', 's_a3', 's_a4', 's_a5',
    's_3di_state',
    't_rsa', 't_phi', 't_psi', 't_a1', 't_a2', 't_a3', 't_a4', 't_a5',
    't_3di_state'
]

#Categorical features
cat_features = [
    's_ss8', 's_3di_letter',
    't_ss8', 't_3di_letter'
]

print("Label distribution:")
print(df[label_cols].sum().sort_values(ascending=False))

Label distribution:
HBOND        1055929
VDW           737061
PIPISTACK      38283
IONIC          35391
PICATION        8885
SSBOND          2100
PIHBOND         1790
dtype: int64


In [10]:
#avoiding data leakage
pdb_ids = df['pdb_id'].unique().to_numpy()
print(f"Total unique PDBs: {len(pdb_ids)}")


train_pdbs, temp_pdbs = train_test_split(pdb_ids, test_size=0.3, random_state=42)
val_pdbs, test_pdbs   = train_test_split(temp_pdbs, test_size=0.50, random_state=42)

train_df = df[df['pdb_id'].isin(train_pdbs)].reset_index(drop=True)
val_df   = df[df['pdb_id'].isin(val_pdbs)].reset_index(drop=True)
test_df  = df[df['pdb_id'].isin(test_pdbs)].reset_index(drop=True)

print(f"Train: {len(train_df)} rows, {len(train_pdbs)} PDBs")
print(f"Val:   {len(val_df)} rows, {len(val_pdbs)} PDBs")
print(f"Test:  {len(test_df)} rows, {len(test_pdbs)} PDBs")

Total unique PDBs: 3827
Train: 999083 rows, 2678 PDBs
Val:   238810 rows, 574 PDBs
Test:  214518 rows, 575 PDBs


Before trusting the splits: : because your task is multi-label and imbalanced, make sure the three splits have similar class prevalences and label overlaps.

In [ ]:
for name, Y in {
    "train": train_df[label_cols],
    "validation": val_df[label_cols],
    "test": test_df[label_cols]
}.items():
    print(name)
    print(Y.mean().sort_values(ascending=False))
    print()

train
HBOND        0.727701
VDW          0.507033
PIPISTACK    0.026643
IONIC        0.024673
PICATION     0.006091
SSBOND       0.001346
PIHBOND      0.001323
dtype: float64

validation
HBOND        0.727294
VDW          0.506566
PIPISTACK    0.024530
IONIC        0.022955
PICATION     0.006277
SSBOND       0.001809
PIHBOND      0.000888
dtype: float64

test
HBOND        0.723529
VDW          0.510540
PIPISTACK    0.027065
IONIC        0.024515
PICATION     0.006065
SSBOND       0.001506
PIHBOND      0.001193
dtype: float64



In [13]:
for name, Y in {
    "train": train_df[label_cols],
    "validation": val_df[label_cols],
    "test": test_df[label_cols]
    }.items():

    overlap_counts = Y.T @ Y
    label_totals = Y.sum(axis=0)
    conditional_overlap = overlap_counts.div(label_totals, axis=0)
    print(f"Conditional overlap for {name} set:")
    display(conditional_overlap)

Conditional overlap for train set:


,HBOND,VDW,IONIC,PIPISTACK,PICATION,SSBOND,PIHBOND
HBOND,1.000000,0.342593,0.031430,0.009859,0.002080,0.000063,0.000483
VDW,0.491695,1.000000,0.025959,0.025004,0.006355,0.001735,0.001577
IONIC,0.927018,0.533469,1.000000,0.000000,0.000000,0.000000,0.000000
PIPISTACK,0.269281,0.475826,0.000000,1.000000,0.000000,0.000000,0.011082
PICATION,0.248480,0.529006,0.000000,0.000000,1.000000,0.000000,0.042564
SSBOND,0.034201,0.653532,0.000000,0.000000,0.000000,1.000000,0.000000
PIHBOND,0.265507,0.604387,0.000000,0.223147,0.195915,0.000000,1.000000


Conditional overlap for validation set:


,HBOND,VDW,IONIC,PIPISTACK,PICATION,SSBOND,PIHBOND
HBOND,1.000000,0.340703,0.029041,0.009149,0.002027,0.000035,0.000253
VDW,0.489159,1.000000,0.024229,0.023212,0.006588,0.002108,0.001033
IONIC,0.920102,0.534659,1.000000,0.000000,0.000000,0.000000,0.000000
PIPISTACK,0.271253,0.479344,0.000000,1.000000,0.000000,0.000000,0.008365
PICATION,0.234823,0.531688,0.000000,0.000000,1.000000,0.000000,0.036691
SSBOND,0.013889,0.590278,0.000000,0.000000,0.000000,1.000000,0.000000
PIHBOND,0.207547,0.589623,0.000000,0.231132,0.259434,0.000000,1.000000


Conditional overlap for test set:


,HBOND,VDW,IONIC,PIPISTACK,PICATION,SSBOND,PIHBOND
HBOND,1.000000,0.343812,0.031383,0.010199,0.002133,0.000071,0.000445
VDW,0.487244,1.000000,0.025694,0.025055,0.006483,0.002045,0.001315
IONIC,0.926222,0.535083,1.000000,0.000000,0.000000,0.000000,0.000000
PIPISTACK,0.272649,0.472615,0.000000,1.000000,0.000000,0.000000,0.009301
PICATION,0.254420,0.545734,0.000000,0.000000,1.000000,0.000000,0.032283
SSBOND,0.034056,0.693498,0.000000,0.000000,0.000000,1.000000,0.000000
PIHBOND,0.269531,0.562500,0.000000,0.210938,0.164062,0.000000,1.000000


In [14]:
os.makedirs("classification_ring/data/processed", exist_ok=True)

train_df.to_parquet("classification_ring/data/processed/train.parquet", index=False)
val_df.to_parquet("classification_ring/data/processed/val.parquet", index=False)
test_df.to_parquet("classification_ring/data/processed/test.parquet", index=False)

print("Splits saved successfully.")
print("DO NOT touch test.parquet until final evaluation pliz!")

Splits saved successfully.
DO NOT touch test.parquet until final evaluation pliz!
